# DiffusionGemma vs Gemma 4 Colab Runner

This notebook is optimized for VS Code with a Google Colab kernel. The first smoke loop does not download models. It only proves that the remote kernel can see the repository, run unit tests, write results, and execute the preflight/phase CLI.

## 1. Runtime Check

Run this cell first after selecting the Colab kernel. It prints Python, current directory, GPU information, and whether the notebook appears to be running inside Colab.

In [ ]:
import os, pathlib, platform, subprocess, sys

def run(cmd):
    print(f"$ {' '.join(cmd)}")
    completed = subprocess.run(cmd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    return completed.returncode

IN_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').exists()
print('python:', sys.version)
print('platform:', platform.platform())
print('cwd:', pathlib.Path.cwd())
print('in_colab_like_runtime:', IN_COLAB)
run(['nvidia-smi'])

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
platform: Linux-6.6.122+-x86_64-with-glibc2.35
cwd: /content
in_colab_like_runtime: True
$ nvidia-smi
Tue Jun 23 19:19:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             47W /  600W |       0MiB /  97887MiB |   

0

## 2. Attach Repository

If the Colab runtime already sees the repository, this cell uses it. If not, upload `dist/diffusion_gemma_bench_source.zip` created locally by:

`python scripts/make_colab_bundle.py`

The uploaded bundle is unpacked to `/content/diffusion_gemma_bench`.

In [3]:
# import os, pathlib, shutil, sys, zipfile

# CANDIDATES = [
#     pathlib.Path.cwd(),
#     pathlib.Path('/content/diffusion_gemma_bench'),
# ]

# def is_repo_root(path):
#     return (path / 'run.py').exists() and (path / 'src').is_dir() and (path / 'tests').is_dir()

# PROJECT_ROOT = next((path for path in CANDIDATES if is_repo_root(path)), None)

# if PROJECT_ROOT is None:
#     try:
#         from google.colab import files
#     except Exception as exc:
#         raise RuntimeError('Repository is not visible and google.colab upload API is unavailable.') from exc
#     print('Upload dist/diffusion_gemma_bench_source.zip from your local workspace.')
#     uploaded = files.upload()
#     zip_names = [name for name in uploaded if name.endswith('.zip')]
#     if not zip_names:
#         raise RuntimeError('No zip file uploaded.')
#     PROJECT_ROOT = pathlib.Path('/content/diffusion_gemma_bench')
#     if PROJECT_ROOT.exists():
#         shutil.rmtree(PROJECT_ROOT)
#     PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
#     with zipfile.ZipFile(zip_names[0]) as zf:
#         zf.extractall(PROJECT_ROOT)

# os.chdir(PROJECT_ROOT)
# if str(PROJECT_ROOT) not in sys.path:
#     sys.path.insert(0, str(PROJECT_ROOT))

# print('PROJECT_ROOT:', PROJECT_ROOT)
# print('repo files:', sorted(p.name for p in PROJECT_ROOT.iterdir())[:20])

## 3. Local Harness Smoke

These checks should pass before we attempt any model download or vLLM server startup.

In [7]:
import subprocess, sys

checks = [
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'],
    [sys.executable, 'run.py', '--profile', 'auto', '--phase', 'preflight'],
    [sys.executable, 'run.py', '--profile', 'q6_core_native', '--phase', 'smoke'],
    [sys.executable, 'run.py', '--profile', 'q6_core_native', '--phase', 'report'],
]

for cmd in checks:
    print('\n$ ' + ' '.join(cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise SystemExit(completed.returncode)


$ /usr/bin/python3 -m unittest discover -s tests


----------------------------------------------------------------------
Ran 0 tests in 0.000s

NO TESTS RAN



SystemExit: 5

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [8]:
!ls

sample_data


## 4. Inspect Smoke Artifacts

This cell shows the machine-readable outputs from the first smoke loop. `smoke_status.json` should say `PENDING_COLAB_BACKEND_GATE`; that is expected until the real vLLM capability gate is implemented.

In [ ]:
import json, pathlib

for path in ['results/preflight.json', 'results/smoke_status.json', 'reports/final_report.md']:
    p = pathlib.Path(path)
    print('\n###', p)
    if p.suffix == '.json':
        print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:4000])
    else:
        print(p.read_text(encoding='utf-8')[:2000])

## 5. Later GPU Phases

After the first smoke loop works, the next repository step is to replace the placeholder smoke phase with the real vLLM capability gate, model artifact checks, and tiny model-load prompts.